# Titian — Provenance for PySpark DataFrames and Python UDFs

The same record-level tracing, driven entirely from Python — including through an
opaque **Python UDF** (Titian re-threads record identity across the UDF's batched
execution) and a **cached DataFrame** (a provenance source boundary).

In [ ]:
import os, sys, glob

ROOT = os.environ.get("BIGASTERISK_HOME") or os.path.abspath("..")

# Jars: a source checkout has them under modules/*/target, the Docker image under jars/.
JARS = sorted(glob.glob(f"{ROOT}/modules/*/target/scala-2.13/bigasterisk-*.jar")) \
    or sorted(glob.glob(f"{ROOT}/jars/bigasterisk-*.jar"))
if not JARS:
    raise SystemExit("No BigAsterisk jars found. Run: bin/sbt package")

FASTUTIL_JAR = os.environ.get("FASTUTIL_JAR") or next(iter(sorted(
    glob.glob(f"{ROOT}/jars/fastutil*.jar")
    + glob.glob(os.path.expanduser("~/Library/Caches/Coursier/**/fastutil-8.5.15.jar"), recursive=True)
    + glob.glob(os.path.expanduser("~/.cache/coursier/**/fastutil-8.5.15.jar"), recursive=True)
)), None)
if not FASTUTIL_JAR:
    raise SystemExit("fastutil jar not found. Run: bin/sbt package")

SPARK_JARS = ",".join(JARS + [FASTUTIL_JAR])
DATA = f"{ROOT}/modules/spark4/src/test/resources"
sys.path.insert(0, f"{ROOT}/python")
print("jars:", *[os.path.basename(j) for j in JARS + [FASTUTIL_JAR]])


In [ ]:
from pyspark.sql import SparkSession
import bigasterisk

spark = (bigasterisk.configure(SparkSession.builder)
    .master("local[2]")
    .appName("titian-notebook")
    .config("spark.jars", SPARK_JARS)
    .config("spark.sql.adaptive.skewJoin.enabled", "false")
    .config("spark.ui.enabled", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

t = bigasterisk.lineage(spark)
t.enable_capture()

In [ ]:
sales = spark.read.schema("category STRING, amount INT").csv(f"{DATA}/sales_parts")
sales.createOrReplaceTempView("sales")
sales.show(5)

## Tracing through a Python UDF

`plus_fee` is opaque Python code executed in worker subprocesses; Titian still traces
each aggregate back to its exact source rows.

In [ ]:
from pyspark.sql.functions import col, udf

plus_fee = udf(lambda a: a + 7, "int")
df = (sales.withColumn("amt2", plus_fee(col("amount")))
           .groupBy("category").sum("amt2"))
out = t.collect_with_lineage(df)
totals = {r["category"]: r["sum(amt2)"] for r, _ in out}
print(totals)
assert totals["groceries"] == 355 + 4 * 7

In [ ]:
g_id = next(i for r, i in out if r["category"] == "groceries")
witnesses = t.trace(df, [g_id]).go_back().show(full=True)
for w in witnesses:
    print(w)
assert sorted(w["amount"] for w in witnesses) == [70, 80, 95, 110]

## Cached DataFrames are source boundaries

Traces over a query that reads `df.cache()` stop at — and `show()` re-reads — the
cached rows.

In [ ]:
t.disable_capture()
cached = spark.table("sales").cache()
cached.count()  # materialize without capture
cached.createOrReplaceTempView("sales_cached")
t.enable_capture()

cdf = spark.sql("SELECT category, SUM(amount) AS total FROM sales_cached GROUP BY category")
cout = t.collect_with_lineage(cdf)
elec_id = next(i for r, i in cout if r["category"] == "electronics")
cw = t.trace(cdf, [elec_id]).go_back().show()
assert sorted(w["amount"] for w in cw) == [310, 380, 420, 99999]
print("cache-boundary witnesses:", cw)

In [ ]:
t.release_lineage(df)
t.release_lineage(cdf)
spark.stop()
print("PYSPARK NOTEBOOK OK")